# Mehrotra, Levenstein et al. (2024): the PoSub claims, on dandiset 000939

Paper: Mehrotra, Levenstein et al. (2024), Current Biology 34, 3043-3054,
[doi:10.1016/j.cub.2024.05.048](https://doi.org/10.1016/j.cub.2024.05.048). The registry's target is
dandiset 000987 (the paper's RSC data, `reproduce_mehrotra_2024.ipynb`). The paper's postsubiculum
(PoSub) recordings were deposited in another dandiset, which the paper cites as DANDI:000939
(Duszkiewicz et al. 2024); the claims file marks the claims on them `data: elsewhere`. They are
tested here, and counted apart from the claims on 000987.

Dataset: [DANDI:000939](https://dandiarchive.org/dandiset/000939/0.261016.2310), version
`0.261016.2310`, streamed (only units, sleep states and epochs are read). The paper cites no
version. The most recent published version is pinned: its release notes (12 May 2026) say the
sleep scoring indexing was fixed and unit ids added, and its units hold the EX flag and waveform
position this analysis uses. The dandiset holds 31 mice; the paper used 16. The 16 sessions are those
for which the paper's code (github.com/PeyracheLab/MehrotraLevenstein_2023 at commit
4ee34f34793c147ac08bf5fffe44ff03d17f8e35) holds DOWN and UP event files (`Analysis/OutputFiles`),
written by detectDOWNs_PoSub.py; these event files are the study's DOWN and UP states and are read
from the repository at that commit.

Claims file: `claims/000987/10.1016-j.cub.2024.05.048.yaml`.

| Claim | Here |
|---|---|
| fig5c-posub-pmr-onset | tested |
| fig5d-isomap-pmr | tested |
| fig1e-delay-range | tested |
| fig1g-dv-sequence-du | tested |
| figs1i-du-more-stable | tested |
| figs1kl-not-firing-rate | tested |
| fig1h-ventral-down-longer | tested |
| fig1b-no-dv-tuning-gradient | tested |
| figs1j-dorsal-ventral-same-hd | tested |
| the other PoSub claims (HD tuning, Isomap ring, LFP) | blocked, see the attempt record |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from scipy.stats import kendalltau, pearsonr, wilcoxon, mannwhitneyu
from sklearn.manifold import Isomap
import pynapple as nap
import mehrotra as m

print("pynapple", nap.__version__, "| dandiset", m.POSUB_DANDISET, m.POSUB_VERSION, "| code", m.CODE_COMMIT[:7])
S = m.load_all_posub()
assets939 = {a["path"].split("/")[0].replace("sub-", ""): a for a in m.list_assets(m.POSUB_DANDISET, m.POSUB_VERSION)}
print(len(S), "sessions loaded")

## Data check

Units, the EX flag, sleep states and epochs of the 16 sessions, and the DOWN and UP intervals of
the paper's event files. The file's NREM epochs (`sleep_states`) are compared with the home cage
epochs and with the end of the recording; the analysis uses NREM within the home cage epochs, as
detectDOWNs_PoSub.py intersects the sleep scoring with the sleep epochs. The event files and the NWB spike times must share a clock: the population
rate inside the DOWN states is compared with the rate in the same intervals shifted by 0.5 s and with
the NREM mean, and the fraction of DOWN-state time inside the file's NREM epochs is given.

In [ ]:
rows = []
for s in S:
    sp, u = s["spikes"], s["units"]
    nrem = s["nrem_hc"]
    last_spike = max(sp[k].t[-1] for k in sp.keys() if len(sp[k]))
    nrem_rate = sum(len(sp[k].restrict(nrem)) for k in sp.keys()) / nrem.tot_length()
    d = s["down"]
    rate = {}
    for shift in (0.0, 0.5):
        ep = nap.IntervalSet(start=d["start"] + shift, end=d["end"] + shift)
        rate[shift] = sum(len(sp[k].restrict(ep)) for k in sp.keys()) / ep.tot_length()
    rows.append(dict(session=s["session"], file_identifier=s["identifier"], units=len(sp), ex=int(u.ex.sum()),
                     fs=int(u.fs.sum()), hd=int(u.hd.sum()), ex_and_fs=int((u.ex & u.fs).sum()),
                     ex_ttp_range=(round(u.trough_to_peak[u.ex].min(), 3), round(u.trough_to_peak[u.ex].max(), 3)),
                     position_range_um=(u.position.min(), u.position.max()),
                     nrem_file_s=round(s["nrem"].tot_length()), nrem_in_home_cage_s=round(nrem.tot_length()),
                     nrem_after_last_spike_s=round(s["nrem"].intersect(nap.IntervalSet(start=last_spike, end=1e7)).tot_length()),
                     last_spike_s=round(last_spike), n_down=len(d), n_up=len(s["up"]),
                     down_in_nrem=round(float(d.intersect(s["nrem"]).tot_length() / d.tot_length()), 3),
                     down_in_home_cage=round(float(d.intersect(s["home_cage"]).tot_length() / d.tot_length()), 3),
                     pop_rate_nrem=round(nrem_rate, 1), pop_rate_down=round(rate[0.0], 1),
                     pop_rate_down_shift_0p5s=round(rate[0.5], 1),
                     epochs=",".join(s["epochs"].tag), unsorted_units=s["spike_unsorted"]))
chk = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(chk.to_string())
print("\nsessions", len(chk), "| units", chk.units.sum(), "(paper 1,657) | EX", chk.ex.sum(), "(paper 1,097)")
print("electrode rel_y spacing (um):", sorted(set(np.round(np.diff(S[0]['rel_y']), 2))),
      "| range", S[0]['rel_y'].min(), "to", S[0]['rel_y'].max())
print("DOWN durations (ms), median", 1000 * float(np.median(np.concatenate([s['down']['end'] - s['down']['start'] for s in S]))),
      "| min", 1000 * float(np.min(np.concatenate([s['down']['end'] - s['down']['start'] for s in S]))),
      "| max", 1000 * float(np.max(np.concatenate([s['down']['end'] - s['down']['start'] for s in S]))))

### DOWN and UP states of A3708-200317b

The event file of session A3708-200317b does not share the clock of the NWB spikes: the population
rate inside its DOWN states equals the rate in the same intervals shifted by 0.5 s and the NREM mean,
and only a quarter of its DOWN-state time falls in the file's NREM epochs. Below, shifts of the
event times from -6000 s to +6000 s in 1 s steps and from -20 s to +20 s in 5 ms steps are tried
(only shifted intervals that stay inside the recording are counted); none brings the rate inside the
DOWN states near the rates of the other sessions' event files (0.0 to 136.7 Hz in the table above). As a check, DOWN and UP states are also
re-detected from the spikes of every session with the procedure of detectDOWNs_RSC.py on the file's
NREM epochs within the home cage epochs, with the UP states restricted to NREM as detectDOWNs_PoSub.py does
(`mehrotra.detect_down_states`, `restrict_up=True`). detectDOWNs_PoSub.py further refines sleep with
the accelerometer and bins each NREM epoch separately; that is not repeated, so the re-detection is
close to, not the same as, the study's.

Event sources used below. Main: the event files, with A3708-200317b's DOWN and UP states
re-detected because its file fails the clock check. Variants: (i) the event files as they are,
(ii) the main source without A3708-200317b, (iii) re-detected states for all sessions.

In [ ]:
s8 = [s for s in S if s["session"] == "A3708-200317b"][0]
allt = np.sort(np.concatenate([s8["spikes"][k].t for k in s8["spikes"].keys()]))
def rate_shift(d, shift):
    st, en = d["start"] + shift, d["end"] + shift
    inside = (st >= allt[0]) & (en <= allt[-1])   # only intervals that stay inside the recording
    st, en = st[inside], en[inside]
    return (np.searchsorted(allt, en) - np.searchsorted(allt, st)).sum() / (en - st).sum()
for grid in (np.arange(-6000, 6001, 1.0), np.arange(-20, 20, 0.005)):
    r = np.array([rate_shift(s8["down"], x) for x in grid])
    print(f"shifts {grid[0]:.0f} to {grid[-1]:.0f} s: lowest rate inside DOWN {r.min():.1f} Hz at {grid[r.argmin()]:.3f} s; median {np.median(r):.1f} Hz")

RED = {}
for s in S:
    RED[s["session"]] = m.detect_down_states(s["spikes"], s["nrem_hc"], restrict_up=True)
rows = []
for s in S:
    d, u = RED[s["session"]]
    sp = s["spikes"]
    rd = sum(len(sp[k].restrict(d)) for k in sp.keys()) / d.tot_length()
    rows.append(dict(session=s["session"], n_down_file=len(s["down"]), n_down_redetected=len(d),
                     file_matched_by_redetected=round(m.interval_match(s["down"], d)["fraction"], 3),
                     redetected_matched_by_file=round(m.interval_match(d, s["down"])["fraction"], 3),
                     pop_rate_in_redetected_down=round(rd, 1)))
print(pd.DataFrame(rows).to_string())

## Measures per session

For every session and event source: the DU-PETH of every unit as Figure5C.py (`mehrotra.du_peth` on
the UP states), UP onset delay and PMR (`mehrotra.pmr_onset`); the PETH around DOWN starts and the
UD timing as Figure1G.py (`mehrotra.peth_events`, `mehrotra.ud_time`); NREM firing rate. For the
main source, the same measures are also computed on two halves of the session for
figs1i-du-more-stable.

In [ ]:
def halves(iset, how):
    n = len(iset)
    a = (np.arange(n) < n // 2) if how == "first_second" else (np.arange(n) % 2 == 0)
    return [nap.IntervalSet(start=iset["start"][k], end=iset["end"][k]) for k in (a, ~a)]

def events_for(s, source):
    if source == "re-detected" or (source == "main" and s["session"] == "A3708-200317b"):
        return RED[s["session"]]
    return s["down"], s["up"]

def session_measures(s, down, up, with_halves=False):
    sp, u = s["spikes"], s["units"]
    p = m.du_peth(sp, up)
    r = m.pmr_onset(p)
    udt = m.ud_time(m.peth_events(sp, down["start"], up))
    df = u.copy()
    df["session"] = s["session"]; df["unit"] = df.index
    df["entered"] = r.entered.values; df["onset"] = r.onset.values; df["pmr"] = r.pmr.values
    df["ud"] = udt.reindex(df.index).values
    df["nrem_rate"] = m.rate_in(sp, s["nrem_hc"]).values
    hr = []
    if with_halves:
        for how in ("first_second", "odd_even"):
            hu, hd = halves(up, how), halves(down, how)
            for k in (0, 1):
                ph = m.pmr_onset(m.du_peth(sp, hu[k]))
                uh = m.ud_time(m.peth_events(sp, hd[k]["start"], hu[k]))
                hr.append(pd.DataFrame(dict(session=s["session"], how=how, half=k, unit=u.index, ex=u.ex.values,
                                            onset=ph.onset.values, ud=uh.reindex(u.index).values)))
    return df, p, hr

ALL, PETHS, half_rows = {}, {}, []
for source in ("main", "event files as they are", "re-detected"):
    rows = []
    for s in S:
        down, up = events_for(s, source)
        df, p, hr = session_measures(s, down, up, with_halves=(source == "main"))
        rows.append(df); half_rows += hr
        if source == "main":
            PETHS[s["session"]] = p
    ALL[source] = pd.concat(rows, ignore_index=True)
ALL["main without A3708-200317b"] = ALL["main"][ALL["main"].session != "A3708-200317b"]
U = ALL["main"]
HALF = pd.concat(half_rows, ignore_index=True)
EX = U[U.ex]
print("units", len(U), "| EX", len(EX), "| EX entering (crossing 0.5 within 0 to 155 ms):", int(EX.entered.sum()),
      "| EX left out:", int((~EX.entered).sum()))
print("EX units with a UD time:", int(EX.ud.notna().sum()))
for k, v in ALL.items():
    e = v[v.ex]
    print(f"{k}: EX {len(e)}, entering {int(e.entered.sum())}, with UD time {int(e.ud.notna().sum())}")

## Claim fig5c-posub-pmr-onset

> In PoSub, PMR is negatively correlated with UP onset delay; earlier-activating EX units show a
> larger overshoot, and virtually no late-activating unit has a large PMR (Kendall tau = -0.19,
> n = 1,096, p = 8.34e-20).

Criterion: as fig5f-rsc-pmr-onset with the PoSub DOWN states of the event files and EX units only;
reproduced when tau is negative with p < 0.05 and within 0.019 of -0.19.

In [ ]:
E = EX[EX.entered]
t, p = kendalltau(E.onset, E.pmr)
print(f"EX units entering: {len(E)} (paper 1,096) | Kendall tau = {t:.4f}, p = {p:.3g} | within 0.019 of -0.19: {abs(t + 0.19) <= 0.019}")
tA, pA = kendalltau(U[U.entered].onset, U[U.entered].pmr)
print(f"variant, all units: n = {int(U.entered.sum())}, tau = {tA:.4f}, p = {pA:.3g}")
late = E[E.onset >= 0.075]
print(f"late units (onset >= 75 ms): {len(late)}, PMR max {late.pmr.max():.3f}, fraction with PMR > 1.5: {(late.pmr > 1.5).mean():.3f}"
      f" | early units (onset < 25 ms): {int((E.onset < 0.025).sum())}, fraction with PMR > 1.5: {(E[E.onset < 0.025].pmr > 1.5).mean():.3f}")
var = []
for k, v in ALL.items():
    e = v[v.ex & v.entered]
    tt, pp = kendalltau(e.onset, e.pmr)
    var.append(dict(events=k, units=len(e), tau=round(tt, 4), p=pp, within_tol=abs(tt + 0.19) <= 0.019))
print(pd.DataFrame(var).to_string())
per = []
for sname, d in E.groupby("session"):
    tt, pp = kendalltau(d.onset, d.pmr)
    per.append(dict(session=sname, n=len(d), tau=round(tt, 3), p=pp))
per = pd.DataFrame(per); print(per.to_string())
print("sessions with negative tau:", int((per.tau < 0).sum()), "of", len(per), "| negative and p < 0.05:", int(((per.tau < 0) & (per.p < 0.05)).sum()))
print("onset range (s):", E.onset.min(), "to", E.onset.max(), "| PMR range:", round(E.pmr.min(), 3), "to", round(E.pmr.max(), 3))

**Result: reproduced, resting on an assumption.** With the main event source (the event files, with
the states of A3708-200317b re-detected), 1,096 of the 1,097 EX units enter, as in the paper, and
Kendall tau between UP onset delay and PMR is -0.177 (p = 3.7e-17), within 0.019 of -0.19. Without
A3708-200317b tau is -0.195 (1,049 units). With the event file of A3708-200317b as it is, tau is
-0.126 (p = 2.0e-9), outside the tolerance, and with all states re-detected it is -0.077. Tau is
negative in all 16 sessions, with p < 0.05 in 7. None of the 46 units with an onset of 75 ms or more
has a PMR above 1.5 (largest 1.32), against 39% of the 198 units with an onset below 25 ms. The outcome
rests on replacing the states of A3708-200317b, whose event file does not share the clock of the spikes.

## Claim fig5d-isomap-pmr

> An Isomap projection of the DU-aligned PETHs of all PoSub EX units shows a continuous gradation of
> profiles whose first dimension correlates strongly with PMR (Pearson R = 0.88, n = 1,096).

Criterion: sklearn Isomap, 2 components, 50 neighbours, on the PETHs of the units of fig5c; the
absolute Pearson R of dimension 1 with PMR; the Methods window (-250 to +250 ms) is the main
result, the code's (0 to 155 ms) a variant. Reproduced when |R| is within 0.088 of 0.88 and
p < 0.05 under the Methods window.

In [ ]:
keys = list(zip(E.session, E.unit))
out = []
proj = {}
for label, lo, hi in (("Methods, -250 to +250 ms", -0.2501, 0.2501), ("code, 0 to 155 ms", -1e-9, 0.1551)):
    X = np.vstack([PETHS[s_].loc[lo:hi, u_].values for s_, u_ in keys])
    Y = Isomap(n_components=2, n_neighbors=50).fit_transform(X)
    proj[label] = Y
    r1, p1 = pearsonr(Y[:, 0], E.pmr.values)
    r2, p2 = pearsonr(Y[:, 1], E.pmr.values)
    ro, po = pearsonr(Y[:, 1], E.onset.values)
    out.append(dict(window=label, bins=X.shape[1], units=X.shape[0], R_dim1_pmr=round(r1, 4), abs_R=round(abs(r1), 4), p=p1,
                    within_tol=abs(abs(r1) - 0.88) <= 0.088, R_dim2_pmr=round(r2, 3), R_dim2_onset=round(ro, 3)))
print(pd.DataFrame(out).to_string())
Y = proj["Methods, -250 to +250 ms"]
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(Y[:, 0], Y[:, 1], c=E.pmr.values, s=4, cmap="OrRd", vmin=0.5, vmax=3)
ax[0].set_title("Isomap of DU-PETHs, colour PMR 0.5 to 3"); ax[0].set_xlabel("dim 1"); ax[0].set_ylabel("dim 2")
ax[1].scatter(Y[:, 0], E.pmr.values, s=4); ax[1].set_xlabel("dim 1"); ax[1].set_ylabel("PMR")
plt.tight_layout(); plt.show()
print("dim 1 range", Y[:, 0].min().round(2), "to", Y[:, 0].max().round(2), "| dim 2 range", Y[:, 1].min().round(2), "to", Y[:, 1].max().round(2),
      "| PMR above the colour scale (3):", int((E.pmr > 3).sum()))

**Result: reproduced.** On the PETHs of the 1,096 units of fig5c-posub-pmr-onset, the first Isomap
dimension correlates with PMR with |R| = 0.865 (p printed as 0) over -250 to +250 ms, the Methods
window, within 0.088 of 0.88; over the code's window, 0 to 155 ms, |R| = 0.875. The second dimension
correlates with UP onset delay (R = -0.535 and -0.562).

## Claim fig1e-delay-range

> PoSub neurons activate with reliable delays after the DU transition, ranging from 5 to 100 ms
> (99th percentile range).

Criterion: the UP onset delays of the EX units of fig5c, pooled; readings (a) 0.5th to 99.5th
percentiles and (b) 1st to 99th; reproduced when under one reading both bounds are within 10% of
5 ms and 100 ms (4.5 to 5.5 ms and 90 to 110 ms). Delays are on the 5 ms grid of the PETH bins.

In [ ]:
d_ms = 1000 * E.onset.values
for label, lo, hi in (("(a) 0.5 to 99.5", 0.5, 99.5), ("(b) 1 to 99", 1, 99)):
    a, b = np.percentile(d_ms, [lo, hi])
    print(f"{label}: {a:.2f} to {b:.2f} ms | lower within 4.5-5.5: {4.5 <= a <= 5.5} | upper within 90-110: {90 <= b <= 110}")
    for method in ("lower", "higher", "nearest"):
        a2, b2 = np.percentile(d_ms, [lo, hi], method=method)
        print(f"     percentile method {method}: {a2:.1f} to {b2:.1f}")
vals, cnts = np.unique(d_ms.round(1), return_counts=True)
print("delay histogram (ms: units):", dict(zip(vals.tolist(), cnts.tolist())))

**Result: reproduced.** Under reading (b), the 1st and 99th percentiles of the 1,096 UP onset delays
are 5 and 100 ms, the paper's range. Under reading (a), the 0.5th and 99.5th percentiles are 0 and
110 ms, and the lower bound is outside 4.5 to 5.5 ms. The delays lie on the 5 ms grid of the PETH, and
the percentile method changes neither reading.

## Claim fig1g-dv-sequence-du

> At the DOWN-to-UP transition PoSub EX units activate in sequence, dorsal units entering the UP
> state before ventral ones; the per-session correlation of UP onset delay with depth is
> consistently non-zero for DU and not for UD transitions.

Criterion: per session, Kendall tau between UP onset delay and depth of EX units (DU), and between
the UD timing of Figure1G.py and depth (UD); DU coefficients differ from 0 by a two-sided one-sample
Wilcoxon test at p < 0.05 with the sign of dorsal-first, UD do not (p >= 0.05), and DU and UD differ
by a Mann-Whitney U test at p < 0.05.

Position is the probe `rel_y` of the unit's peak electrode (12.5 um steps along the shank); larger
`rel_y` is read as dorsal, the convention of probe geometry (y increasing from the tip), and
Figure1H.py's `pd.cut(depth, 2, labels=[1, 0])` with "0 is dorsal" also puts larger values dorsal.
Dorsal-first is then a negative tau between delay and position, as in Figure 1F (tau = -0.6).

In [ ]:
def dv_table(V):
    g = []
    for sname, d in V[V.ex].groupby("session", sort=False):
        a = d[d.entered]
        t_du, p_du = kendalltau(a.onset, a.position)
        b = d[d.ud.notna()]
        t_ud, p_ud = kendalltau(b.ud, b.position) if b.ud.nunique() > 1 else (np.nan, np.nan)
        g.append(dict(session=sname, n_du=len(a), tau_du=t_du, p_du=p_du, n_ud=len(b), ud_distinct_values=b.ud.nunique(),
                      tau_ud=t_ud, p_ud=p_ud))
    return pd.DataFrame(g)

def dv_tests(G):
    du, ud = G.tau_du.dropna(), G.tau_ud.dropna()
    w_du, w_ud, mw = wilcoxon(du), wilcoxon(ud), mannwhitneyu(du, ud)
    return dict(n_du=len(du), median_du=round(du.median(), 3), negative_du=int((du < 0).sum()), W_du=w_du.statistic, p_du=w_du.pvalue,
                n_ud=len(ud), median_ud=round(ud.median(), 3), W_ud=w_ud.statistic, p_ud=w_ud.pvalue, U=mw.statistic, p_U=mw.pvalue,
                holds=bool(du.median() < 0 and w_du.pvalue < 0.05 and w_ud.pvalue >= 0.05 and mw.pvalue < 0.05))

G = dv_table(U)
print(G.round(4).to_string())
print("\npaper: DU W = 1, p = 6.1e-5; UD W = 38, p = 0.13; DU against UD U = 52, p = 0.005")
print(pd.DataFrame({k: dv_tests(dv_table(v)) for k, v in ALL.items()}).T.to_string())
print("sessions with DU tau negative and p < 0.05 (main):", int(((G.tau_du < 0) & (G.p_du < 0.05)).sum()))

**Result: reproduced, resting on an assumption.** With the main event source, the per-session tau
between UP onset delay and position is negative in 15 of 16 sessions (median -0.309) and differs from 0
(one-sample Wilcoxon W = 1, p = 6.1e-5, the paper's values); the UD tau does not (W = 38, p = 0.13,
the paper's values); and DU and UD differ (Mann-Whitney U = 51, p = 0.0039; paper U = 52, p = 0.005).
A negative tau is dorsal-first when larger `rel_y` is dorsal. With the event file of A3708-200317b as
it is, or without that session, the UD tau differs from 0 (p = 0.048) and the claim would hold only in
part; with all states re-detected it holds (UD p = 0.13; U p = 0.048). The outcome rests on the
re-detected states of A3708-200317b and on the direction of `rel_y`.

## Claim figs1i-du-more-stable

> Across halves of a session, the delays of PoSub units at UP state onset are more stable than their
> timing at the UD transition (Wilcoxon signed-rank test, W = 4, n = 16, p = 0.0002).

Criterion: per session, the DU delays and UD timings of EX units computed separately in two halves
of the session, the correlation of each across units between halves, and DU against UD compared
over sessions by a two-sided Wilcoxon signed-rank test. Readings: halves as the first and second
half of the NREM DOWN and UP states, or odd and even ones; Pearson or Kendall correlation.
Reproduced when DU exceeds UD at p < 0.05 under all readings. A unit enters a session's correlation
when it has a value in both halves.

In [ ]:
res_i = []
per_i = []
for how in ("first_second", "odd_even"):
    for corr_name, corr in (("Pearson", pearsonr), ("Kendall", kendalltau)):
        du_c, ud_c = [], []
        for sname, d in HALF[(HALF.how == how) & HALF.ex].groupby("session", sort=False):
            h0 = d[d.half == 0].set_index("unit"); h1 = d[d.half == 1].set_index("unit")
            ok = h0.onset.notna() & h1.onset.notna()
            du_c.append(corr(h0.onset[ok], h1.onset[ok])[0])
            ok2 = h0.ud.notna() & h1.ud.notna()
            ok_ud = ok2.sum() > 2 and h0.ud[ok2].nunique() > 1 and h1.ud[ok2].nunique() > 1
            ud_c.append(corr(h0.ud[ok2], h1.ud[ok2])[0] if ok_ud else np.nan)
            per_i.append(dict(how=how, corr=corr_name, session=sname, n_du=int(ok.sum()), du=round(du_c[-1], 3),
                              n_ud=int(ok2.sum()), ud=round(ud_c[-1], 3)))
        du_c, ud_c = np.array(du_c), np.array(ud_c)
        keep = ~np.isnan(du_c) & ~np.isnan(ud_c)
        du_c, ud_c = du_c[keep], ud_c[keep]
        w = wilcoxon(du_c, ud_c)
        res_i.append(dict(halves=how, correlation=corr_name, sessions=int(keep.sum()), median_du=round(np.median(du_c), 3),
                          median_ud=round(np.median(ud_c), 3), du_gt_ud_sessions=int((du_c > ud_c).sum()),
                          W=w.statistic, p=w.pvalue, du_exceeds_ud_p05=bool(np.median(du_c - ud_c) > 0 and w.pvalue < 0.05)))
print(pd.DataFrame(res_i).to_string())
print(pd.DataFrame(per_i).pivot_table(index="session", columns=["how", "corr"], values=["du", "ud"]).round(2).to_string())

**Result: blocked, methods underspecified.** DU delays are more stable across halves of the session
than UD timings under three readings: first and second half with Kendall's tau (DU larger in 13 of 16
sessions, W = 14, p = 0.0034), and odd and even DOWN states with Pearson's R or Kendall's tau (14 of
14, W = 0, p = 0.00012). Under the fourth, first and second half with Pearson's R, they are not (8 of
16, W = 51, p = 0.40). The criterion blocks the claim when the readings disagree. With odd and even
halves, no UD correlation could be computed in two sessions, which are left out.

## Claim figs1kl-not-firing-rate

> UP onset delays of PoSub EX units are explained by DV position better than by NREM firing rate;
> per animal, the regression coefficients of delay on position exceed those on firing rate
> (Wilcoxon signed-rank, W = 19, n = 16, p = 0.009) and differ from 0 (one-sample Wilcoxon, W = 0,
> n = 16, p = 3.052e-5).

Criterion: per animal, a linear regression of the UP onset delays of EX units on DV position and
NREM firing rate together. Readings: predictors z-scored or raw; "better" comparing absolute or
signed coefficients. Reproduced when under all readings the position coefficients exceed the
firing-rate coefficients (two-sided Wilcoxon signed-rank, p < 0.05) and differ from 0 (two-sided
one-sample Wilcoxon, p < 0.05). NREM firing rate is the unit's rate in the file's NREM epochs
within the home cage epochs.
Position is entered as dorsoventral distance from the top of the probe (larger is ventral), so that
a dorsal-first sequence gives a positive coefficient.

In [ ]:
coef = []
for sname, d in EX[EX.entered].groupby("session", sort=False):
    ventral = d.position.max() - d.position  # um from the most dorsal unit
    for scale in ("z-scored", "raw"):
        X = np.column_stack([ventral, d.nrem_rate])
        if scale == "z-scored":
            X = (X - X.mean(0)) / X.std(0)
        X1 = np.column_stack([np.ones(len(d)), X])
        b = np.linalg.lstsq(X1, d.onset.values, rcond=None)[0]
        coef.append(dict(session=sname, scale=scale, n=len(d), b_position=b[1], b_rate=b[2]))
C = pd.DataFrame(coef)
rows = []
for scale in ("z-scored", "raw"):
    c = C[C.scale == scale]
    for comp in ("absolute", "signed"):
        bp, br = (c.b_position.abs(), c.b_rate.abs()) if comp == "absolute" else (c.b_position, c.b_rate)
        w1 = wilcoxon(bp, br); w0 = wilcoxon(c.b_position)
        rows.append(dict(scale=scale, compare=comp, median_b_position=c.b_position.median(), median_b_rate=c.b_rate.median(),
                         position_gt_rate=int((bp > br).sum()), W_pos_vs_rate=w1.statistic, p_pos_vs_rate=w1.pvalue,
                         W_pos_vs_0=w0.statistic, p_pos_vs_0=w0.pvalue,
                         holds=bool(np.median(bp - br) > 0 and w1.pvalue < 0.05 and w0.pvalue < 0.05)))
print(pd.DataFrame(rows).to_string())
print(C[C.scale == "z-scored"].round(4).to_string())

**Result: blocked, methods underspecified.** With z-scored predictors and signed coefficients, the
position coefficients exceed the firing-rate coefficients (15 of 16 sessions, W = 9, p = 0.0010) and
differ from 0 (W = 0, p = 3.1e-5), as the paper reports (W = 19, p = 0.009; W = 0, p = 3.052e-5). With
z-scored predictors and absolute coefficients the first test does not reach p < 0.05 (12 of 16, W = 32,
p = 0.065). With raw predictors the coefficients per um of position and per Hz of rate are on
different scales, and the position coefficient is the larger in absolute value in 0 of 16 sessions.
The criterion blocks the claim when the readings disagree.

## Claim fig1h-ventral-down-longer

> DOWN states detected from ventral PoSub last longer than those detected from dorsal PoSub
> (Wilcoxon signed-rank test, W = 9, n = 16, p = 0.01).

Criterion: as Figure1H.py and the Methods. Units split into dorsal and ventral halves by depth (two
equal-width depth bins); the pooled spikes of each half form a PETH around each DOWN start (spikes
restricted to the UP states), normalized by its mean rate in UP states and smoothed as for the DU
PETH; the DOWN duration of a half is the first time after +5 ms where the normalized rate exceeds
0.2 minus the last time between -150 and -5 ms where it exceeds 0.2. Units: all units (the code
uses all units with the gd flag; the files have no gd flag) and EX units only (the Methods' general
rule). Reproduced when ventral exceeds dorsal by a two-sided Wilcoxon signed-rank test at p < 0.05
under both; blocked as methods_underspecified when they disagree.

In [ ]:
def half_duration(peth_col, thr=0.2):
    lags = np.round(peth_col.index.values, 6)
    post = peth_col[(lags >= 0.005) & (peth_col.values > thr)]
    pre = peth_col[(lags >= -0.150) & (lags <= -0.005) & (peth_col.values > thr)]
    if len(post) == 0 or len(pre) == 0:
        return np.nan
    return post.index[0] - pre.index[-1]

dur = []
for s in S:
    u = s["units"]
    for sel_name, mask in (("all units", np.ones(len(u), bool)), ("EX units", u.ex.values)):
        uu = u[mask]
        level = pd.cut(uu.position, 2, labels=["ventral", "dorsal"])  # larger rel_y is dorsal
        mua = {}
        for code, lab in ((0, "dorsal"), (1, "ventral")):
            idx = uu.index[level == lab]
            mua[code] = nap.Ts(t=np.sort(np.concatenate([s["spikes"][k].t for k in idx])))
        g = nap.TsGroup(mua)
        down, up = events_for(s, "main")
        q = m.peth_events(g, down["start"], up, window=0.255)
        q.columns = ["dorsal", "ventral"]
        dur.append(dict(session=s["session"], units=sel_name, n_dorsal=int((level == "dorsal").sum()),
                        n_ventral=int((level == "ventral").sum()),
                        dorsal_ms=1000 * half_duration(q["dorsal"]), ventral_ms=1000 * half_duration(q["ventral"])))
D = pd.DataFrame(dur)
print(D.round(1).to_string())
for sel_name, d in D.groupby("units"):
    d = d.dropna(subset=["dorsal_ms", "ventral_ms"])
    w = wilcoxon(d.dorsal_ms, d.ventral_ms)
    print(f"{sel_name}: median dorsal {d.dorsal_ms.median():.1f} ms, ventral {d.ventral_ms.median():.1f} ms; ventral longer in "
          f"{int((d.ventral_ms > d.dorsal_ms).sum())} of {len(d)}, equal in {int((d.ventral_ms == d.dorsal_ms).sum())}; "
          f"Wilcoxon W = {w.statistic:.1f}, p = {w.pvalue:.3g} (paper W = 9, p = 0.01)")

**Result: reproduced.** DOWN states detected from the ventral half are longer than those from the
dorsal half with all units (median 107.5 against 97.5 ms; longer in 12 of 16 sessions, equal in 2;
W = 12, p = 0.011; paper W = 9, p = 0.01) and with EX units only (117.5 against 102.5 ms; W = 3,
p = 0.0019). Durations lie on the 5 ms grid of the PETH.

## Claim fig1b-no-dv-tuning-gradient

> HD information, linear speed and angular head velocity tuning of PoSub EX cells do not vary with DV
> position; resultant vector length and peak rate show weak but significant correlations with it.

Criterion: a claim of absence for HD information, speed and AHV scores, judged by the paper's
test, a two-sided Kendall tau against depth at p < 0.05 over all EX cells; the measures as the
Methods define them (`mehrotra.hd_metrics`, which cites the passage). Reproduced when those three
are not significant and resultant vector length and peak rate are positively correlated with p <
0.05. Position is `rel_y` (larger is dorsal, as for fig1g), so "positively correlated" is read in
that direction, as Figure1H.py's depth variable (larger is dorsal) suggests; the paper's sign
convention for Figure S1 is not stated, and the opposite direction is reported too.

In [ ]:
hm = []
for s in S:
    e = s["epochs"][s["epochs"].tag == "wake_square"]
    ep = nap.IntervalSet(start=e.start.values, end=e.end.values)
    r = m.hd_metrics(assets939[s["session"].split("-")[0]], s["spikes"], ep).join(s["units"])
    r["session"] = s["session"]
    hm.append(r)
HM = pd.concat(hm)
HX = HM[HM.ex]
rows = []
for col, paper in (("hd_info", "tau = -0.03, p = 0.13"), ("hd_info_per_spike", "variant of HD information"),
                   ("speed_score", "tau = -0.03, p = 0.18"), ("ahv_score", "tau = -0.02, p = 0.23"),
                   ("rvl", "tau = 0.063, p = 0.002"), ("peak_rate", "tau = 0.075, p = 0.0002")):
    d = HX[[col, "position"]].dropna()
    t, p = kendalltau(d[col], d.position)
    rows.append(dict(measure=col, n=len(d), tau_vs_position_dorsal_positive=round(t, 4), p=p, significant=p < 0.05, paper=paper,
                     range=(round(d[col].min(), 3), round(d[col].max(), 3))))
print(pd.DataFrame(rows).to_string())
print("EX cells:", len(HX), "| EX cells flagged HD:", int(HX.hd.sum()), "| wake_square duration (s) per session, min to max:",
      round(min(s["epochs"][s["epochs"].tag == "wake_square"].eval("end - start").sum() for s in S)), "to",
      round(max(s["epochs"][s["epochs"].tag == "wake_square"].eval("end - start").sum() for s in S)))

**Result: reproduced, resting on an assumption.** Over the 1,097 EX cells, speed scores (tau = 0.029,
p = 0.15) and AHV scores (tau = 0.035, p = 0.081) do not vary with position, while resultant vector
length (tau = 0.061, p = 0.0026) and peak rate (tau = 0.077, p = 0.00014) increase toward larger
`rel_y`, close to the paper's values (0.063, p = 0.002; 0.075, p = 0.0002). HD information in bits per
spike does not vary with position (tau = 0.034, p = 0.099); computed as the formula is printed, in
bits per second, it does (tau = 0.082, p = 5.1e-5). The outcome rests on reading HD information in
bits per spike, the unit in which the Methods use it to select HD cells for Isomap ("HD information
exceeding 0.2 bits/spike", citing Duszkiewicz et al.).

## Claim figs1j-dorsal-ventral-same-hd

> During NREM sleep, dorsal and ventral PoSub represent the same HD; the decoded angular difference
> is not different from that with shuffled cell depths.

Criterion: a claim of absence. Bayesian decoding (`mehrotra.decode_hd`, as pynapple decode_1d) in
50 ms UP-state bins with at least 5 HD cells spiking in each half; per session, the summary of the
absolute angular difference between the dorsal and ventral decoded HD, z-scored against 100
shuffles of cell depth; reproduced when the two-sided one-sample Wilcoxon test of the z-scores
against 0 gives p >= 0.05. Readings: halves by equal-width depth bins or by the median depth; the
summary as the mean or the median. HD cells are the file's `is_head_direction` flag; tuning curves
from the square arena; UP states of the main event source. In each shuffle the bin selection (5
cells in each half) is applied anew to the shuffled halves.

In [ ]:
rng = np.random.default_rng(20240531)
zrows = []
for s in S:
    u = s["units"]
    hdc = u.index[u.hd].tolist()
    e = s["epochs"][s["epochs"].tag == "wake_square"]
    ep = nap.IntervalSet(start=e.start.values, end=e.end.values)
    tc = m.hd_tuning_curves(assets939[s["session"].split("-")[0]], s["spikes"][hdc], ep)
    down, up = events_for(s, "main")
    C = s["spikes"][hdc].count(m.DECODE_BIN, up).values
    pos = u.loc[hdc, "position"].values
    splits = {"equal-width": np.asarray(pd.cut(pos, 2, labels=["ventral", "dorsal"])),
              "median": np.where(pos > np.median(pos), "dorsal", "ventral")}
    def stat(labels):
        dm, vm = labels == "dorsal", labels == "ventral"
        ok = ((C[:, dm] > 0).sum(1) >= m.DECODE_MIN_CELLS) & ((C[:, vm] > 0).sum(1) >= m.DECODE_MIN_CELLS)
        if ok.sum() < 10:
            return np.nan, np.nan, int(ok.sum())
        ad = m.decode_hd(C[ok][:, dm], tc.values[:, dm], tc.index.values)
        av = m.decode_hd(C[ok][:, vm], tc.values[:, vm], tc.index.values)
        d = np.abs(np.angle(np.exp(1j * (ad - av))))
        return d.mean(), np.median(d), int(ok.sum())
    for split, lab in splits.items():
        obs = stat(lab)
        sh = np.array([stat(rng.permutation(lab)) for _ in range(100)])
        for k, summ in ((0, "mean"), (1, "median")):
            shk = sh[:, k][np.isfinite(sh[:, k])]
            z = (obs[k] - shk.mean()) / shk.std() if len(shk) > 1 and shk.std() > 0 else np.nan
            zrows.append(dict(session=s["session"], split=split, summary=summ, hd_cells=len(hdc),
                              dorsal=int((lab == "dorsal").sum()), ventral=int((lab == "ventral").sum()),
                              bins=obs[2], observed=obs[k], shuffle_mean=shk.mean() if len(shk) else np.nan, z=z))
Z = pd.DataFrame(zrows)
res = []
for (split, summ), d in Z.groupby(["split", "summary"]):
    zz = d.z.dropna()
    w = wilcoxon(zz)
    res.append(dict(split=split, summary=summ, sessions=len(zz), median_z=round(zz.median(), 3), positive=int((zz > 0).sum()),
                    W=w.statistic, p=w.pvalue, no_difference=w.pvalue >= 0.05))
print(pd.DataFrame(res).to_string())
print("\npaper: W = 38, n = 16, p = 0.13")
print(Z[(Z.split == "equal-width") & (Z.summary == "mean")].round(3).to_string())
print("bins kept per session (equal-width), min to max:", Z[Z.split == "equal-width"].bins.min(), "to", Z[Z.split == "equal-width"].bins.max())

**Result: reproduced.** Under all four readings the z-scored dorsal-ventral angular difference does
not differ from that with shuffled depths: equal-width halves, mean W = 40, p = 0.46 and median W = 31,
p = 0.19 (14 sessions); median split, W = 59, p = 0.98 and W = 56, p = 0.82 (15 sessions). With
equal-width halves, A3709-200601 (7 bins) and A3713-200909a (0 bins) have fewer than 10 bins with 5 HD
cells spiking in each half and are left out. The paper reports W = 38, p = 0.13 over 16 sessions.

## Summary

| Claim | Outcome |
|---|---|
| fig5c-posub-pmr-onset | reproduced, resting on an assumption (tau = -0.177, n = 1,096) |
| fig5d-isomap-pmr | reproduced (abs R = 0.865) |
| fig1e-delay-range | reproduced (5 to 100 ms, 1st to 99th percentile) |
| fig1g-dv-sequence-du | reproduced, resting on an assumption |
| figs1i-du-more-stable | blocked, methods underspecified (3 of 4 readings hold) |
| figs1kl-not-firing-rate | blocked, methods underspecified (1 of 4 readings holds) |
| fig1h-ventral-down-longer | reproduced |
| fig1b-no-dv-tuning-gradient | reproduced, resting on an assumption (HD information in bits per spike) |
| figs1j-dorsal-ventral-same-hd | reproduced |

How the analysis differs from the paper's. The DOWN and UP states are the paper's event files, read
from its code at commit 4ee34f3, except for A3708-200317b, re-detected from the spikes because its
event file does not share the clock of the NWB spikes. Units are classified by the file's
`is_excitatory` flag (the code also requires a `gd` flag the files do not hold; the counts match the
paper's). Unit position is the `rel_y` of the peak electrode, read as larger toward dorsal. NREM is the
file's NREM epochs within the home cage epochs, as detectDOWNs_PoSub.py intersects sleep scoring with
the sleep epochs. Readings the paper leaves open were all run; the ones the outcomes rest on are named
in each result.

What was found about the dataset (numbers above):

- The 16 sessions hold 1,657 units, 1,097 of them flagged excitatory, the paper's counts; 1,096 of the
  EX units enter the PMR test, as in Figure 5C.
- The event file of A3708-200317b in the paper's code does not fit the NWB spikes of that session: the
  population rate inside its DOWN states (454.7 Hz) equals the rate in the same intervals shifted by
  0.5 s (455.8 Hz), and no shift from -6,000 to +6,000 s brings it below 309 Hz, where the other
  sessions' DOWN states give 0.0 to 136.7 Hz.
- The NREM epochs (`sleep_states`) lie partly outside the home cage epochs, and in A3711-200810b,
  A3717-201021 and A3723-201115 they run 3,508 to 5,092 s past the last spike.
- The electrode table gives `x`, `y` and `z` as NaN and does not say in which direction `rel_y` runs.